# Correlations Between All Variables

Outcomes, engagement, demographics and survey controls, offboarding sample.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg

FIG_DIR = "../tables_figures/"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "mathtext.fontset": "custom",
    "mathtext.rm": "Arial",
    "mathtext.it": "Arial:italic",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

activity = pd.read_csv("../data/offboarding_sample_round1.csv", sep=";", decimal=",", encoding="utf-8-sig")
print(activity.shape)
# Keep only the days a participant could have been active: from their first day (where their
# rows start) up to and including the day they received the offboarding survey
activity = activity[activity["day"] <= activity["offboarding_received_date"]]
print(activity.shape)

In [ ]:
outcome_vars = ["user_satisfaction_1", "user_satisfaction_2", "user_satisfaction_3",
                "topics", "parties"]
control_vars = ['sd_age_years', 'sd_gender', 'sd_edu_be_rec',
                'interest_politics_', 'news_interest_', 'internet_use', 'news_use_6',
                'ideo_lr_1', 'div_values_mean', 'div_traits_mean', 'div_perceived__2',
                'div_perceived__3', 'inst_trust_1', 'inst_trust_2']

# Same engagement window as RQ1: up to and including the day the offboarding survey was received
in_window = activity[activity["day"] <= activity["offboarding_received_date"]]

# Survey answers repeat on every day row, so take one value per participant
participants = (
    activity.groupby("handle")[["arm"] + outcome_vars + control_vars].first()
    .join(in_window.groupby("handle")[["active", "reactions"]].sum()
          .rename(columns={"active": "n_active_days", "reactions": "n_reactions"}))
)
participants["woman"] = participants["sd_gender"].map({1: 0, 2: 1})  # 'Other' becomes missing
participants["edu_level"] = participants["sd_edu_be_rec"].map({1: 1, 2: 1, 3: 1, 4: 2, 5: 2, 6: 3, 7: 3})
participants.shape

In [ ]:
LABELS = {
    "n_active_days": "Active Days",
    "n_reactions": "Reactions",
    "user_satisfaction_1": "Would Use the Feed Again",
    "user_satisfaction_2": "News Quality in the Feed",
    "user_satisfaction_3": "Satisfied with News Selection",
    "topics": "Perceived Topic Diversity",
    "parties": "Perceived Party Diversity",
    "sd_age_years": "Age",
    "woman": "Gender (Woman)",
    "edu_level": "Education",
    "interest_politics_": "Political Interest",
    "news_interest_": "News Interest",
    "internet_use": "Internet Use",
    "news_use_6": "News Use via Social Media",
    "ideo_lr_1": "Left–Right Ideology",
    "div_values_mean": "Diversity-Seeking Values",
    "div_traits_mean": "Diversity-Seeking Behavior",
    "div_perceived__2": "Perceived Topic Diversity in Media",
    "div_perceived__3": "Perceived Party Diversity in Media",
    "inst_trust_1": "Trust in Social Media",
    "inst_trust_2": "Trust in News Media",
}

corr = (
    participants[list(LABELS)]
    .corr(method="spearman")
    .rename(index=LABELS, columns=LABELS)
)

pairs = pg.pairwise_corr(participants[list(LABELS)], method="spearman", padjust="fdr_bh")
print(len(pairs), "pairwise correlations")
pvals = (
    pairs.pivot(index="Y", columns="X", values="p-corr")
    .reindex(index=list(LABELS), columns=list(LABELS))
    .rename(index=LABELS, columns=LABELS)
)

def stars(p):
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""


FONT_FAMILY = "Arial"
LABEL_FONT_SIZE = 8.5
ANNOT_FONT_SIZE = 7

fig, ax = plt.subplots(figsize=(7, 6.5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
annot = (corr.apply(lambda col: col.map(
             lambda r: f"{r:.2f}".replace("0.", ".").replace("-", "\u2212")))
         + pvals.apply(lambda col: col.map(stars)))
# Generate the heatmap with legibility controls
sns.heatmap(
    corr,
    annot=annot,  # Display numbers inside the cells
    fmt="",  # Force exactly 2 decimal places (APA standard)
    cmap="RdBu_r",  # Colorblind-safe, grayscale-friendly palette
    vmin=-1,
    vmax=1,  # Set colorbar limits explicitly
    mask=mask, 
    square=True,  # Force cells to be perfectly square
    cbar_kws={"label": "Spearman's ρ", "shrink": 0.7},  # Shrink colorbar slightly to match plot height
    annot_kws={
        "size": ANNOT_FONT_SIZE,
        "family": FONT_FAMILY,
        "weight": "normal"
    },  # Format cell numbers
    ax=ax,
)
ax.tick_params(length=0, labelsize=LABEL_FONT_SIZE)
fig.tight_layout()
fig.savefig(FIG_DIR + "correlation_matrix_all.png")
plt.show()

In [ ]:
# see where p-corr is less than 0.05 
pairs[pairs['p-corr'] <= 0.05].sort_values('X')